# Laboratorium 2: Współbieżność i Równoległość w Pythonie
### Skoroszyt Edukacyjny - Wersja dla Studentów

---

## 1. Wstęp: Koncepcja "Wielu Zadań"

Zanim zaczniemy pisać kod, musimy rozróżnić dwa kluczowe pojęcia:

1. **Współbieżność (Concurrency)**: Wykonywanie wielu zadań "na zmianę". Wyobraź sobie kelnera, który obsługuje 5 stolików. Nie robi wszystkiego naraz, ale szybko przełącza się między nimi. Dla klientów wygląda to, jakby obsługiwał ich równocześnie.
2. **Równoległość (Parallelism)**: Wykonywanie wielu zadań faktycznie w tym samym momencie. To sytuacja, w której mamy 5 kelnerów i każdy obsługuje jeden stolik.

W Pythonie współbieżność realizujemy najczęściej za pomocą **Wątków (Threads)**, a równoległość za pomocą **Procesów (Processes)**.

---

## 2. Wielowątkowość (Threading) - Zadania I/O-bound

Wątki są idealne, gdy program większość czasu spędza na **czekaniu** na odpowiedź z sieci (zapytania HTTP). W tym czasie procesor się nudzi – wątki pozwalają mu wysłać kolejne zapytania, nie czekając na poprzednie.

---

### Demo: Scraping Kalendarza Kulturalnego (Krakow.pl)

**Kod zawarty w poniższych komórkach (analogicznie do plików `lab_2_1_demo.py` oraz `lab_2_2_demo.py`) pozwala na pobieranie tytułów wydarzeń kulturalnych z oficjalnego kalendarium miasta Krakowa (krakow.pl).** 

Przykładowy adres źródłowy: `https://www.krakow.pl/kalendarium/1919,shw,2026-03-20,0,day.html`. 

Demo pokazuje proces pobierania danych z 5 kolejnych stron tego zestawienia:
1. **Wersja sekwencyjna**: Zadanie wykonywane jest krok po kroku, co pozwala zaobserwować sumaryczny czas oczekiwania na każde z zapytań HTTP z osobna (wysoki koszt operacji wejścia/wyjścia).
2. **Optymalizacja**: Kod zostaje zmodyfikowany z użyciem modułu `concurrent.futures`, wykorzystując `ThreadPoolExecutor`.

Dzięki temu zapytania sieciowe są wysyłane równolegle, co drastycznie skraca czas całkowity działania programu, demonstrując praktyczną przewagę wielowątkowości w zadaniach typu **I/O-bound** (zależnych od odpowiedzi sieciowej).

In [1]:
import requests
from bs4 import BeautifulSoup
import time

def download_site(url):
    """Pobiera jedną stronę i wyciąga tytuły wydarzeń."""
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'html.parser')
    event_titles = [item.text.strip() for item in soup.select('.item__link h3')]
    return event_titles

def run_sequential_demo():
    date_str = "2026-03-20"
    base_url = "https://www.krakow.pl/kalendarium/1919,shw"
    sites = [f"{base_url},{date_str},{i},day.html" for i in range(5)]
    
    print(f"Rozpoczynam pobieranie SEKWENCYJNE 5 stron...")
    start = time.time()
    
    all_titles = []
    for url in sites:
        all_titles.extend(download_site(url))
        
    print(f"Pobrano łącznie {len(all_titles)} tytułów.")
    print("Pierwsze 10 wyników:")
    for i, title in enumerate(all_titles[:10], 1):
        print(f"{i}. {title}")
        
    print(f"\nCzas wykonania: {time.time() - start:.2f}s")

run_sequential_demo()

Rozpoczynam pobieranie SEKWENCYJNE 5 stron...
Pobrano łącznie 100 tytułów.
Pierwsze 10 wyników:
1. Seks po krakosku
2. Tango (Teatr Maska)
3. Hańba! Hioba Dylana w Zaścianku
4. Strasznie śmieszne...
5. Masażystka (MOS Underground)
6. Poważne Niepoważne – koncert piosenek Ewy Ryks
7. Zwiedzanie teatru z przewodnikiem
8. Wrona: Spacery po piekle w Klubie Spotkań Poczta Główna
9. Dom weselny
10. Klaun Feliks - Ale cyrki 2+

Czas wykonania: 2.76s


In [2]:
import concurrent.futures

def run_threaded_demo():
    date_str = "2026-03-20"
    base_url = "https://www.krakow.pl/kalendarium/1919,shw"
    sites = [f"{base_url},{date_str},{i},day.html" for i in range(5)]
    
    print(f"Rozpoczynam pobieranie WIELOWĄTKOWE 5 stron...")
    start = time.time()
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
        results = list(executor.map(download_site, sites))
    
    all_titles = [title for sublist in results for title in sublist]
    
    print(f"Pobrano łącznie {len(all_titles)} tytułów.")
    print("Pierwsze 10 wyników:")
    for i, title in enumerate(all_titles[:10], 1):
        print(f"{i}. {title}")
        
    print(f"\nCzas wykonania (wątki): {time.time() - start:.2f}s")

run_threaded_demo()

Rozpoczynam pobieranie WIELOWĄTKOWE 5 stron...
Pobrano łącznie 100 tytułów.
Pierwsze 10 wyników:
1. Seks po krakosku
2. Tango (Teatr Maska)
3. Hańba! Hioba Dylana w Zaścianku
4. Strasznie śmieszne...
5. Masażystka (MOS Underground)
6. Poważne Niepoważne – koncert piosenek Ewy Ryks
7. Zwiedzanie teatru z przewodnikiem
8. Wrona: Spacery po piekle w Klubie Spotkań Poczta Główna
9. Dom weselny
10. Klaun Feliks - Ale cyrki 2+

Czas wykonania (wątki): 0.63s


--- 
## 3. Synchronizacja: Problem Hazardu i Lock

Gdy wiele wątków próbuje zmieniać tę samą zmienną w tym samym momencie (np. saldo na koncie), dochodzi do tzw. **Race Condition** (wyścigu). Rozwiązaniem jest **Lock** (blokada).

In [3]:
import threading

class BankAccount:
    def __init__(self):
        self.balance = 0
        self.lock = threading.Lock()

    def deposit(self, amount):
        with self.lock:
            current = self.balance
            time.sleep(0.0001) # Symulacja opóźnienia
            self.balance = current + amount

account = BankAccount()
with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
    executor.map(lambda _: account.deposit(1), range(100))
    
print(f"Saldo końcowe: {account.balance} zł (oczekiwano: 100)")

Saldo końcowe: 100 zł (oczekiwano: 100)


--- 
## 4. Wieloprocesowość (Multiprocessing) - Zadania CPU-bound

Kiedy musimy wykonać ciężkie obliczenia matematyczne (np. szukanie liczb pierwszych), wątki nam nie pomogą. Musimy użyć osobnych procesów.

**Ważne (macOS/Windows)**: Ze względu na metodę `spawn` startu procesów, funkcje pomocnicze (jak `find_primes`) muszą znajdować się w zewnętrznym pliku `.py` (tutaj: `lab2_functions.py`) i być importowane.

In [4]:
import multiprocessing
import time
# Importujemy funkcję z oddzielnego pliku, aby uniknąć błędu spawn na macOS
from lab2_functions import find_primes

def run_primes_demo():
    cores = multiprocessing.cpu_count()
    print(f"Praca na {cores} procesach (rdzeniach)...")
    start = time.time()
    
    limit = 1_000_000
    chunk = limit // cores
    ranges = [(i, i + chunk) for i in range(0, limit, chunk)]

    with multiprocessing.Pool(processes=cores) as pool:
        results = pool.starmap(find_primes, ranges)
    
    print(f"Zakończono w czasie {time.time() - start:.2f}s.")

if __name__ == "__main__":
    run_primes_demo()

Praca na 10 procesach (rdzeniach)...
Zakończono w czasie 0.37s.


---
# Zadania do samodzielnego wykonania

Poniższe zadania należy zrealizować w oparciu o wiedzę zdobytą na laboratoriach oraz instrukcje zawarte w pliku PDF.

### Zadanie 1 (Threading)
Przy użyciu publicznego API **Cat Facts** (`https://catfact.ninja/fact`), które zwraca przy każdym wywołaniu losowy fakt na temat kotów:
1. Pobierz sekwencyjnie 20 faktów i zmierz czas całkowitego działania programu.
2. Zmodyfikuj kod, aby wysyłać zapytania wielowątkowo przy użyciu `ThreadPoolExecutor`.
3. Porównaj czasy wykonania.

*Podpowiedź: Użyj `requests.get(URL).json().get('fact')`*

In [5]:
import requests
import time
import concurrent.futures

CAT_API_URL = "https://catfact.ninja/fact"
NUM_FACTS = 20

def fetch_cat_fact(index):
    """Fetches a single cat fact from the API."""
    response = requests.get(CAT_API_URL)
    fact = response.json().get('fact')
    return fact

# 1. Sequential Execution
print(f"Starting sequential fetch of {NUM_FACTS} cat facts...")
start_time = time.time()
sequential_facts = [fetch_cat_fact(i) for i in range(NUM_FACTS)]
sequential_duration = time.time() - start_time
print(f"Sequential fetch completed in {sequential_duration:.2f}s")

# 2. Threaded Execution
print(f"\nStarting threaded fetch of {NUM_FACTS} cat facts...")
start_time = time.time()
with concurrent.futures.ThreadPoolExecutor(max_workers=10) as executor:
    threaded_facts = list(executor.map(fetch_cat_fact, range(NUM_FACTS)))
threaded_duration = time.time() - start_time
print(f"Threaded fetch completed in {threaded_duration:.2f}s")

# Summary
print(f"\nComparison: Threaded version is {sequential_duration / threaded_duration:.1f}x faster!")
print(f"Sample fact: {threaded_facts[0]}")

Starting sequential fetch of 20 cat facts...
Sequential fetch completed in 4.69s

Starting threaded fetch of 20 cat facts...
Threaded fetch completed in 1.34s

Comparison: Threaded version is 3.5x faster!
Sample fact: The average litter of kittens is between 2 - 6 kittens.


### Zadanie 2 (Wątki i Kolejka - Producent-Konsument)
Napisz program o strukturze **producent-consumers**:
1. **Producent**: Generuje kolejne liczby naturalne i dodaje je do kolejki (`queue.Queue`).
2. **Konsument 1**: Pobiera z kolejki tylko liczby **parzyste**.
3. **Konsument 2**: Pobiera z kolejki tylko liczby **nieparzyste**.

Użyj wątków do realizacji producenta i obu konsumentów. Program powinien zakończyć się po przetworzeniu określonej puli liczb.

In [6]:
import queue
import threading
import time

def producer(data_queue, count):
    """Generates natural numbers and puts them into the queue."""
    for i in range(1, count + 1):
        print(f"[Producer] Generating: {i}")
        data_queue.put(i)
        time.sleep(0.01)
    # Signal end of production for consumers
    data_queue.put(None)
    data_queue.put(None)

def even_consumer(data_queue):
    """Processes only even numbers from the queue."""
    while True:
        item = data_queue.get()
        if item is None:
            break
        if item % 2 == 0:
            print(f"  [Even Consumer] Processed: {item}")
        else:
            # Put it back for the other consumer
            data_queue.put(item)
            time.sleep(0.01)
        data_queue.task_done()

def odd_consumer(data_queue):
    """Processes only odd numbers from the queue."""
    while True:
        item = data_queue.get()
        if item is None:
            break
        if item % 2 != 0:
            print(f"  [Odd Consumer] Processed: {item}")
        else:
            # Put it back for the other consumer
            data_queue.put(item)
            time.sleep(0.01)
        data_queue.task_done()

def run_producer_consumer_demo():
    data_queue = queue.Queue(maxsize=10)
    total_numbers = 20

    threads = [
        threading.Thread(target=producer, args=(data_queue, total_numbers)),
        threading.Thread(target=even_consumer, args=(data_queue,)),
        threading.Thread(target=odd_consumer, args=(data_queue,))
    ]

    for t in threads:
        t.start()

    for t in threads:
        t.join()
    
    print("Production and consumption completed.")

run_producer_consumer_demo()

[Producer] Generating: 1
  [Odd Consumer] Processed: 1
[Producer] Generating: 2
  [Even Consumer] Processed: 2
[Producer] Generating: 3
  [Odd Consumer] Processed: 3
[Producer] Generating: 4
  [Even Consumer] Processed: 4
[Producer] Generating: 5
  [Odd Consumer] Processed: 5
[Producer] Generating: 6
  [Even Consumer] Processed: 6
[Producer] Generating: 7
  [Odd Consumer] Processed: 7
[Producer] Generating: 8
  [Even Consumer] Processed: 8
[Producer] Generating: 9
  [Odd Consumer] Processed: 9
[Producer] Generating: 10
  [Even Consumer] Processed: 10
[Producer] Generating: 11
  [Odd Consumer] Processed: 11
[Producer] Generating: 12
  [Even Consumer] Processed: 12
[Producer] Generating: 13
  [Odd Consumer] Processed: 13
[Producer] Generating: 14
  [Even Consumer] Processed: 14
[Producer] Generating: 15
  [Odd Consumer] Processed: 15
[Producer] Generating: 16
  [Even Consumer] Processed: 16
[Producer] Generating: 17
  [Odd Consumer] Processed: 17
[Producer] Generating: 18
  [Even Consume

### Zadanie 3 (Multiprocessing)
Napisz program, który zrównolegli obliczanie sumy kolejnych stu potęg dla każdej liczby z ciągu liczb naturalnych w dużym zakresie (np. 1 - 10 000).
Użyj modułu `multiprocessing` oraz gotowej funkcji `calculate_power_sum(n)` z pliku `lab2_functions.py`.

Pamiętaj o bezpiecznym uruchamianiu procesów na macOS (`if __name__ == "__main__":`).

In [7]:
import multiprocessing
import time
from lab2_functions import calculate_power_sum

def run_multiprocessing_sum():
    numbers = list(range(1, 10001))
    cores = multiprocessing.cpu_count()
    
    print(f"Starting parallel power sum calculation using {cores} cores...")
    start_time = time.time()
    
    with multiprocessing.Pool(processes=cores) as pool:
        results = pool.map(calculate_power_sum, numbers)
    
    duration = time.time() - start_time
    print(f"Completed calculation of {len(results)} sums in {duration:.4f}s")
    print(f"Sample result (for 5): {results[4]}")

if __name__ == "__main__":
    run_multiprocessing_sum()

Starting parallel power sum calculation using 10 cores...
Completed calculation of 10000 sums in 0.1628s
Sample result (for 5): 9860761315262647567646607066034827870915080438862787559628486633300780
